# 第19章: 強化学習を最新環境で継続検証する

この Notebook は、読み取り専用の原本 `machine-learning-book/ch19/` を参照しながら、第19章の主要アイデアを `pytest --nbmake` で安定実行できる形に再構成したものです。
Gym や GUI rendering への依存は避け、割引報酬、Bellman 方程式、value iteration、Monte Carlo、Q-learning、小さな DQN をローカル完結の GridWorld で確認します。


## この Notebook で確認すること

- `uv` 環境で Chapter 19 の主要パッケージが利用できることを確認する。
- 原本図版を読み取り専用サブモジュールから参照できることを確認する。
- 割引報酬と価値関数の計算を小さな例で確かめる。
- GridWorld に対して value iteration を実装し、Bellman 最適方程式を確認する。
- Monte Carlo と Q-learning を同じ環境で比較する。
- one-hot 状態を入力とする小さな DQN を学習させる。


In [ ]:
from collections import defaultdict, deque
from importlib.metadata import version
from pathlib import Path
import platform
import random
import sys

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import Image, display

SEED = 123
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

REPO_ROOT = next(
    (
        candidate.resolve()
        for candidate in [Path.cwd(), *Path.cwd().parents]
        if (candidate / 'machine-learning-book').exists()
    ),
    None,
)
assert REPO_ROOT is not None, 'machine-learning-book を含むリポジトリルートを見つけられませんでした'

FIG_DIR = REPO_ROOT / 'machine-learning-book/ch19/figures'
assert FIG_DIR.exists(), f'図版ディレクトリが見つかりません: {FIG_DIR}'

print(f'Python 実行ファイル: {sys.executable}')
print(f'Python バージョン: {platform.python_version()}')
print(f'Matplotlib バックエンド: {matplotlib.get_backend()}')
print(f'図版ディレクトリ: {FIG_DIR}')


In [ ]:
PACKAGE_NAMES = ['numpy', 'pandas', 'matplotlib', 'torch', 'pytest', 'nbmake']
package_versions = pd.DataFrame(
    [(name, version(name)) for name in PACKAGE_NAMES],
    columns=['パッケージ', 'バージョン'],
)
package_versions


## 原本図版の参照

原本の MDP、GridWorld、Q-learning、DQN の図版を参照し、Notebook 内の軽量実装と対応づけます。


In [ ]:
selected_figures = [
    ('19_02.png', 440),
    ('19_07.png', 520),
    ('19_09.png', 520),
    ('19_13.png', 520),
]

for figure_name, width in selected_figures:
    figure_path = FIG_DIR / figure_name
    print(figure_path.name)
    display(Image(filename=str(figure_path), width=width))


## 割引報酬の確認

第19章の最初の数式で出てくる return を、小さな報酬列に対して明示的に計算します。


In [ ]:
rewards = np.array([0.0, 0.0, 0.0, 1.0])
gamma = 0.9
returns = np.array(
    [sum((gamma**k) * rewards[t + k] for k in range(len(rewards) - t)) for t in range(len(rewards))]
)
assert abs(returns[0] - 0.729) < 1e-8
pd.DataFrame({'time_step': np.arange(len(rewards)), 'reward': rewards, 'return': returns})


## 小さな MDP に対する Bellman 型の政策評価

`cool`, `warm`, `overheated` の 3 状態だけを持つ小さな MDP で、固定方策の価値を反復評価します。


In [ ]:
state_values = {'cool': 0.0, 'warm': 0.0, 'overheated': 0.0}
transitions = {
    ('cool', 'slow'): [(1.0, 'cool', 1.0)],
    ('cool', 'fast'): [(1.0, 'warm', 2.0)],
    ('warm', 'slow'): [(1.0, 'cool', 1.0)],
    ('warm', 'fast'): [(1.0, 'overheated', -10.0)],
}
policy = {'cool': 'fast', 'warm': 'slow'}

value_history = []
for _ in range(20):
    updated_values = state_values.copy()
    for state in ['cool', 'warm']:
        action = policy[state]
        updated_values[state] = sum(
            probability * (reward + 0.9 * state_values[next_state])
            for probability, next_state, reward in transitions[(state, action)]
        )
    state_values = updated_values
    value_history.append((state_values['cool'], state_values['warm']))

assert state_values['cool'] > state_values['warm']
pd.DataFrame(value_history, columns=['V(cool)', 'V(warm)']).tail()


## 小さな GridWorld と value iteration

原本では Gym ベースの環境と GridWorld を使いますが、ここでは GUI なしで動く最小の deterministic GridWorld を Notebook 内に定義します。
その上で value iteration を実装し、最適価値と最適方策を求めます。


In [ ]:
class GridWorld:
    def __init__(
        self,
        rows: int = 4,
        cols: int = 4,
        start: tuple[int, int] = (0, 0),
        goal: tuple[int, int] = (3, 3),
        trap: tuple[int, int] = (1, 3),
        wall: tuple[int, int] = (1, 1),
        step_reward: float = -0.04,
    ):
        self.rows = rows
        self.cols = cols
        self.start = start
        self.goal = goal
        self.trap = trap
        self.wall = wall
        self.step_reward = step_reward
        self.action_names = ['up', 'right', 'down', 'left']
        self.nA = 4
        self.state_to_idx = {(r, c): r * cols + c for r in range(rows) for c in range(cols)}
        self.idx_to_state = {idx: state for state, idx in self.state_to_idx.items()}
        self.reset()

    def valid_states(self):
        return [state for state in self.state_to_idx if state != self.wall]

    def reset(self):
        self.position = self.start
        return self.state_to_idx[self.position]

    def step(self, action: int):
        row, col = self.position
        moves = [
            (max(row - 1, 0), col),
            (row, min(col + 1, self.cols - 1)),
            (min(row + 1, self.rows - 1), col),
            (row, max(col - 1, 0)),
        ]
        next_row, next_col = moves[action]
        if (next_row, next_col) == self.wall:
            next_row, next_col = row, col
        self.position = (next_row, next_col)

        reward = self.step_reward
        done = False
        if self.position == self.goal:
            reward = 1.0
            done = True
        elif self.position == self.trap:
            reward = -1.0
            done = True
        return self.state_to_idx[self.position], reward, done, {}

    def transitions(self, state_idx: int, action: int):
        state = self.idx_to_state[state_idx]
        if state in (self.goal, self.trap):
            return [(1.0, state_idx, 0.0, True)]
        old_position = self.position
        self.position = state
        next_state, reward, done, _ = self.step(action)
        self.position = old_position
        return [(1.0, next_state, reward, done)]


env = GridWorld()
valid_state_indices = [env.state_to_idx[state] for state in env.valid_states()]


def value_iteration(env: GridWorld, gamma: float = 0.9, tol: float = 1e-6):
    values = np.zeros(env.rows * env.cols)
    for _ in range(200):
        delta = 0.0
        for state_idx in valid_state_indices:
            state = env.idx_to_state[state_idx]
            if state in (env.goal, env.trap):
                continue
            q_values = []
            for action in range(env.nA):
                q_values.append(
                    sum(
                        probability * (reward + gamma * values[next_state] * (not done))
                        for probability, next_state, reward, done in env.transitions(state_idx, action)
                    )
                )
            new_value = max(q_values)
            delta = max(delta, abs(new_value - values[state_idx]))
            values[state_idx] = new_value
        if delta < tol:
            break

    policy = {}
    for state_idx in valid_state_indices:
        state = env.idx_to_state[state_idx]
        if state in (env.goal, env.trap):
            continue
        action_values = [
            sum(
                probability * (reward + gamma * values[next_state] * (not done))
                for probability, next_state, reward, done in env.transitions(state_idx, action)
            )
            for action in range(env.nA)
        ]
        policy[state_idx] = int(np.argmax(action_values))
    return values, policy

optimal_values, optimal_policy = value_iteration(env)
assert optimal_values[env.reset()] > 0.3

value_grid = np.full((env.rows, env.cols), np.nan)
for state_idx in valid_state_indices:
    row, col = env.idx_to_state[state_idx]
    value_grid[row, col] = optimal_values[state_idx]

fig, ax = plt.subplots(figsize=(4, 4))
heatmap = ax.imshow(value_grid, cmap='viridis')
for row in range(env.rows):
    for col in range(env.cols):
        if not np.isnan(value_grid[row, col]):
            ax.text(col, row, f'{value_grid[row, col]:.2f}', ha='center', va='center', color='white', fontsize=9)
ax.set_title('value iteration の状態価値')
fig.colorbar(heatmap, ax=ax, fraction=0.046, pad=0.04)
plt.show()
plt.close(fig)


## Monte Carlo による状態価値推定

探索的な行動を含むエピソードを繰り返し生成し、first-visit Monte Carlo で状態価値を推定します。


In [ ]:
def run_episode(policy_epsilon: float = 0.2, q_table: defaultdict | None = None, max_steps: int = 30):
    state = env.reset()
    episode = []
    for _ in range(max_steps):
        if q_table is None or np.random.rand() < policy_epsilon:
            action = np.random.choice(env.nA)
        else:
            action = int(np.argmax(q_table[state]))
        next_state, reward, done, _ = env.step(action)
        episode.append((state, action, reward, next_state, done))
        state = next_state
        if done:
            break
    return episode

returns_by_state = defaultdict(list)
mc_values = defaultdict(float)
for _ in range(300):
    episode = run_episode(policy_epsilon=0.35)
    G = 0.0
    visited_states = set()
    for state, action, reward, next_state, done in reversed(episode):
        G = reward + 0.9 * G
        if state not in visited_states:
            returns_by_state[state].append(G)
            mc_values[state] = float(np.mean(returns_by_state[state]))
            visited_states.add(state)

pd.DataFrame(
    {
        'state_index': sorted(mc_values.keys())[:8],
        'mc_value': [mc_values[state_idx] for state_idx in sorted(mc_values.keys())[:8]],
    }
)


## Q-learning で GridWorld を解く

原本の GridWorld Q-learning を、同じ環境で GUI なしに再構成します。
探索率を徐々に下げながら、開始状態の Q 値が改善することを確認します。


In [ ]:
q_table = defaultdict(lambda: np.zeros(env.nA))
epsilon = 1.0
episode_returns = []

for episode_idx in range(220):
    state = env.reset()
    total_reward = 0.0
    for _ in range(40):
        if np.random.rand() < epsilon:
            action = np.random.choice(env.nA)
        else:
            action = int(np.argmax(q_table[state]))
        next_state, reward, done, _ = env.step(action)
        td_target = reward + 0.9 * np.max(q_table[next_state]) * (not done)
        q_table[state][action] += 0.2 * (td_target - q_table[state][action])
        total_reward += reward
        state = next_state
        if done:
            break
    epsilon = max(0.05, epsilon * 0.985)
    episode_returns.append(total_reward)

start_state = env.reset()
assert np.max(q_table[start_state]) > 0.3

fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(episode_returns)
ax.set_title('Q-learning のエピソード報酬')
ax.set_xlabel('episode')
ax.set_ylabel('total reward')
ax.grid(alpha=0.3)
plt.show()
plt.close(fig)

pd.DataFrame(
    {
        'action': env.action_names,
        'q_value': q_table[start_state].round(4),
    }
).sort_values('q_value', ascending=False)


## 小さな DQN を同じ GridWorld に適用する

最後に、状態を one-hot ベクトルで表し、小さな replay memory と target network を持つ最小 DQN を学習させます。
Gym の CartPole は使わず、同じ GridWorld に対する近似関数ベースの Q 学習だけを確認します。


In [ ]:
class QNetwork(nn.Module):
    def __init__(self, n_states: int, n_actions: int):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(n_states, 32),
            nn.ReLU(),
            nn.Linear(32, n_actions),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.network(x)


def one_hot_state(state_idx: int, n_states: int) -> torch.Tensor:
    state = torch.zeros(n_states)
    state[state_idx] = 1.0
    return state

n_states = env.rows * env.cols
policy_net = QNetwork(n_states, env.nA)
target_net = QNetwork(n_states, env.nA)
target_net.load_state_dict(policy_net.state_dict())
optimizer = torch.optim.Adam(policy_net.parameters(), lr=0.01)
replay_memory = deque(maxlen=500)
epsilon = 1.0
dqn_loss_history = []

for episode_idx in range(180):
    state = env.reset()
    for _ in range(30):
        if np.random.rand() < epsilon:
            action = np.random.choice(env.nA)
        else:
            with torch.no_grad():
                action = int(torch.argmax(policy_net(one_hot_state(state, n_states))))
        next_state, reward, done, _ = env.step(action)
        replay_memory.append((state, action, reward, next_state, done))
        state = next_state

        if len(replay_memory) >= 32:
            batch = random.sample(replay_memory, 32)
            states = torch.stack([one_hot_state(sample[0], n_states) for sample in batch])
            actions = torch.tensor([sample[1] for sample in batch]).unsqueeze(1)
            rewards = torch.tensor([sample[2] for sample in batch], dtype=torch.float32)
            next_states = torch.stack([one_hot_state(sample[3], n_states) for sample in batch])
            dones = torch.tensor([sample[4] for sample in batch], dtype=torch.float32)

            q_pred = policy_net(states).gather(1, actions).squeeze(1)
            with torch.no_grad():
                q_target = rewards + 0.9 * target_net(next_states).max(dim=1).values * (1.0 - dones)

            loss = F.mse_loss(q_pred, q_target)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            dqn_loss_history.append(float(loss.detach()))

        if done:
            break

    epsilon = max(0.05, epsilon * 0.98)
    if episode_idx % 15 == 0:
        target_net.load_state_dict(policy_net.state_dict())

with torch.no_grad():
    start_q_values = policy_net(one_hot_state(env.reset(), n_states))

assert torch.max(start_q_values).item() > 0.2

fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(dqn_loss_history)
ax.set_title('DQN の学習損失')
ax.set_xlabel('update step')
ax.set_ylabel('MSE loss')
ax.grid(alpha=0.3)
plt.show()
plt.close(fig)

pd.DataFrame({'action': env.action_names, 'q_value': start_q_values.detach().numpy().round(4)})


## まとめ

- 原本の MDP / GridWorld / Q-learning の流れを、GUI や Gym 依存なしの最小環境で再構成しました。
- value iteration、Monte Carlo、Q-learning、DQN を同一環境で比較できるようにしました。
- すべてのセルは `nbmake` のヘッドレス実行を前提にしており、外部ダウンロードや対話入力には依存しません。
